# ResNet-18 compression baseline

Manuscript section 3.5.2: train a ResNet-18 baseline on the harmonized images, then measure how its
accuracy drops under social-media compression (clean vs JPEG 90/70/50/30, WebP, downscaling,
repeated recompression). ΔA = A_clean − A_compressed is the baseline the multi-signal framework
is compared against.

Before starting: **Runtime > Change runtime type > T4 GPU**, and run notebook 01 through step 8 once
so the prepared data is saved on Google Drive.

## 1. Get the code and check the GPU

In [ ]:
!git clone https://github.com/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution.git repo
%cd repo
!pip install -q -e ".[train,dev]"
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (training will be very slow)")

## 2. Restore the prepared data from Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
ZIP = "/content/drive/MyDrive/thesis-data/harmonized.zip"
if not os.path.exists(ZIP):
    raise FileNotFoundError(f"{ZIP} not found: run notebook 01 through step 8 first, it saves the prepared data there")
!unzip -qo $ZIP -d .
!wc -l data/harmonized/train.csv data/harmonized/val.csv data/harmonized/test.csv

## 3. Train ResNet-18
Up to 50 epochs with early stopping (stops after 5 epochs without validation improvement).
Each epoch prints training loss, validation loss and validation accuracy. Checkpoints are
written to Google Drive as well, so a disconnect doesn't lose the model.

In [ ]:
!mkdir -p /content/drive/MyDrive/thesis-data/models && rm -rf outputs/models && ln -s /content/drive/MyDrive/thesis-data/models outputs/models
!python scripts/train.py --arch resnet18

In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open("outputs/models/resnet18/history.json"))
ep = [r["epoch"] for r in h]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(ep, [r["train_loss"] for r in h], label="train"); axes[0].plot(ep, [r["val_loss"] for r in h], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(ep, [r["val_acc"] for r in h]); axes[1].set_title("Validation accuracy"); axes[1].set_xlabel("epoch")
plt.tight_layout()

## 4. Compression stress test
Evaluates the best checkpoint on the test split under every transformation in the config.
Results go to `outputs/stress_test/resnet18/results.csv`.

In [ ]:
!mkdir -p /content/drive/MyDrive/thesis-data/stress_test && rm -rf outputs/stress_test && ln -s /content/drive/MyDrive/thesis-data/stress_test outputs/stress_test
!python scripts/stress_test.py --arch resnet18

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv("outputs/stress_test/resnet18/results.csv")
display(df.round(4))
ax = df.plot.bar(x="condition", y="accuracy", legend=False, figsize=(9, 3.5), ylim=(0, 1))
ax.set_ylabel("test accuracy"); ax.set_title("ResNet-18 accuracy under social-media transforms")
plt.tight_layout()